In [1]:
!pip install spacy scikit-learn nltk pandas matplotlib seaborn wordcloud plotly
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys

project_path = '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM'

sys.path.append(project_path)

In [4]:
import pandas as pd

df = pd.read_csv(f'{project_path}/data/BBC News Train.csv')
df.head()

,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business


In [5]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1490 entries, 0 to 1489
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ArticleId  1490 non-null   int64 
 1   Text       1490 non-null   object
 2   Category   1490 non-null   object
dtypes: int64(1), object(2)
memory usage: 35.1+ KB


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['Category']
)

x_train = train_df.drop(['Category', 'ArticleId'], axis=1)
y_train = train_df['Category']

x_test = test_df.drop(['Category', 'ArticleId'], axis=1)
y_test = test_df['Category']

In [7]:
x_train.head()

,Text
305,winter freeze keeps oil above $50 oil prices c...
783,concerns over windows atms cash machine networ...
197,holmes is hit by hamstring injury kelly holmes...
941,wal-mart is sued over rude lyrics the parents ...
977,nintendo adds media playing to ds nintendo is ...


In [8]:
y_train.head()

,Category
305,business
783,tech
197,sport
941,entertainment
977,tech


In [9]:
x_test.head()

,Text
1439,what high-definition will do to dvds first it ...
815,stars pay tribute to actor davis hollywood sta...
1403,roche turns down federer offer australian te...
1360,ukip candidate suspended eurosceptic party uki...
846,brown and blair face new rift claims for the u...


In [10]:
y_test.head()

,Category
1439,tech
815,entertainment
1403,sport
1360,politics
846,politics


In [11]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

label_encoder.classes_

array(['business', 'entertainment', 'politics', 'sport', 'tech'],
      dtype=object)

In [12]:
from newsbot.preprocessing import clean_text, preprocess_text
from newsbot.td_idf_analysis import get_top_tfidf_terms
from newsbot.pos import analyze_pos_patterns

In [13]:
x_train_copy_tf = x_train.copy()

x_train_copy_tf['Text'] = x_train_copy_tf['Text'].apply(clean_text)
x_train_copy_tf['Text'] = x_train_copy_tf['Text'].apply(preprocess_text)
x_train_copy_tf.head()

,Text
305,winter freeze keep oil oil price carry rise we...
783,concern windows atms cash machine network soon...
197,holmes hit hamstring injury kelly holmes force...
941,walmart sue rude lyric parent yearold girl sue...
977,nintendo add medium play ds nintendo release a...


In [14]:
x_test_copy_tf = x_test.copy()

x_test_copy_tf['Text'] = x_test_copy_tf['Text'].apply(clean_text)
x_test_copy_tf['Text'] = x_test_copy_tf['Text'].apply(preprocess_text)
x_test_copy_tf.head()

,Text
1439,highdefinition dvds humble home video dvd holl...
815,star pay tribute actor davis hollywood star in...
1403,roche turn federer offer australian tennis coa...
1360,ukip candidate suspend eurosceptic party ukip ...
846,brown blair face new rift claim umpteenth time...


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8
    )

tfidf_matrix_train = tfidf_vectorizer.fit_transform(x_train_copy_tf['Text'])

tfidf_matrix_test = tfidf_vectorizer.transform(x_test_copy_tf['Text'])


feature_names = tfidf_vectorizer.get_feature_names_out()

tfidf_df_x_train = pd.DataFrame(tfidf_matrix_train.toarray(), columns=feature_names)
tfidf_df_x_test = pd.DataFrame(tfidf_matrix_test.toarray(), columns=feature_names)


In [16]:
tfidf_df_x_train.head()

,abandon,abbas,ability,able,abn,abolish,abroad,absence,absolute,absolutely,...,youngster,youth,yuan,yugansk,yuganskneftegas,yukos,yukos claim,zealand,zero,zone
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.038717,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.055243,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
tfidf_df_x_test.head()

,abandon,abbas,ability,able,abn,abolish,abroad,absence,absolute,absolutely,...,youngster,youth,yuan,yugansk,yuganskneftegas,yukos,yukos claim,zealand,zero,zone
0,0.0,0.0,0.000000,0.040490,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.043237,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.031676,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.038908,0.000000,0.0,0.0,0.0,0.0,0.0,0.04288,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
x_train_copy_pos = x_train.copy()

x_train_copy_pos['Text'] = x_train_copy_pos['Text'].apply(clean_text)
x_train_copy_pos.head()

,Text
305,winter freeze keeps oil above oil prices carri...
783,concerns over windows atms cash machine networ...
197,holmes is hit by hamstring injury kelly holmes...
941,walmart is sued over rude lyrics the parents o...
977,nintendo adds media playing to ds nintendo is ...


In [19]:
x_test_copy_pos = x_test.copy()

x_test_copy_pos['Text'] = x_test_copy_pos['Text'].apply(clean_text)
x_test_copy_pos.head()

,Text
1439,what highdefinition will do to dvds first it w...
815,stars pay tribute to actor davis hollywood sta...
1403,roche turns down federer offer australian tenn...
1360,ukip candidate suspended eurosceptic party uki...
846,brown and blair face new rift claims for the u...


In [20]:
import spacy
from collections import Counter

nlp = spacy.load('en_core_web_sm')

In [21]:

pos_results_train = []

for idx, row in x_train_copy_pos.iterrows():
    pos_analysis = analyze_pos_patterns(row['Text'])
    pos_results_train.append(pos_analysis)

pos_df_x_train = pd.DataFrame(pos_results_train).fillna(0)


pos_results_test = []

for idx, row in x_test_copy_pos.iterrows():
    pos_analysis = analyze_pos_patterns(row['Text'])
    pos_results_test.append(pos_analysis)

pos_df_x_test = pd.DataFrame(pos_results_test).fillna(0)


In [22]:
pos_df_x_train.head()

,NN,VBZ,IN,NNS,VBN,VBG,NNP,JJ,DT,VBD,...,RBS,XX,LS,SYM,NNPS,WP$,ADD,NFP,.,HYPH
0,0.180921,0.013158,0.180921,0.065789,0.026316,0.029605,0.088816,0.088816,0.101974,0.049342,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.168622,0.021994,0.133431,0.114370,0.030792,0.026393,0.045455,0.071848,0.087977,0.045455,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.126923,0.030769,0.119231,0.046154,0.038462,0.026923,0.061538,0.080769,0.080769,0.061538,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.147059,0.037815,0.109244,0.075630,0.021008,0.033613,0.100840,0.067227,0.096639,0.054622,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.195122,0.052265,0.142857,0.059233,0.038328,0.013937,0.087108,0.041812,0.142857,0.017422,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
pos_df_x_test.head()

,WP,NN,MD,VB,TO,RB,PRP,VBD,DT,JJ,...,JJS,RBR,SYM,:,RBS,NNPS,WP$,FW,XX,HYPH
0,0.004469,0.191061,0.025698,0.054749,0.027933,0.069274,0.051397,0.018994,0.087151,0.044693,...,0.002235,0.002235,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.005362,0.155496,0.005362,0.040214,0.034853,0.040214,0.056300,0.061662,0.067024,0.048257,...,0.005362,0.000000,0.002681,0.002681,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.192513,0.005348,0.037433,0.021390,0.037433,0.026738,0.048128,0.064171,0.101604,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,0.007874,0.136483,0.031496,0.049869,0.013123,0.034121,0.057743,0.057743,0.110236,0.057743,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.009174,0.161206,0.011796,0.038008,0.022280,0.058978,0.045872,0.057667,0.120577,0.082569,...,0.001311,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
x_train_final = pd.concat([tfidf_df_x_train, pos_df_x_train], axis=1)
x_test_final = pd.concat([tfidf_df_x_test, pos_df_x_test], axis=1)

x_test_final = x_test_final.reindex(columns=x_train_final.columns, fill_value=0)

In [25]:
x_train_final.head()


,abandon,abbas,ability,able,abn,abolish,abroad,absence,absolute,absolutely,...,RBS,XX,LS,SYM,NNPS,WP$,ADD,NFP,.,HYPH
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.038717,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.055243,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
x_train_final.shape

(1192, 5042)

In [27]:
y_train.shape

(1192,)

In [28]:
x_test_final.head()

,abandon,abbas,ability,able,abn,abolish,abroad,absence,absolute,absolutely,...,RBS,XX,LS,SYM,NNPS,WP$,ADD,NFP,.,HYPH
0,0.0,0.0,0.000000,0.040490,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0,0.000000,0.0,0.0,0,0,0,0.0
1,0.0,0.0,0.000000,0.043237,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0,0.002681,0.0,0.0,0,0,0,0.0
2,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0,0.000000,0.0,0.0,0,0,0,0.0
3,0.0,0.0,0.000000,0.031676,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,0.0,0,0.000000,0.0,0.0,0,0,0,0.0
4,0.0,0.0,0.038908,0.000000,0.0,0.0,0.0,0.0,0.0,0.04288,...,0.0,0.0,0,0.000000,0.0,0.0,0,0,0,0.0


In [29]:
x_test_final.shape

(298, 5042)

In [30]:
y_test.shape

(298,)

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_validate

logisticRegressor = LogisticRegression(max_iter=1000)
naiveBayes = MultinomialNB()
svm = LinearSVC()

metrics = ['accuracy', 'f1_macro']

models = [logisticRegressor, naiveBayes, svm]

for model in models:
  results = cross_validate(model, x_train_final, y_train_encoded, cv=5, scoring=metrics)

  print(f'Model: {model}')
  print(f'Accuracy: {results["test_accuracy"].mean()}')
  print(f'F1 Score: {results["test_f1_macro"].mean()}')

Model: LogisticRegression(max_iter=1000)
Accuracy: 0.9664217151295664
F1 Score: 0.9656223005183122
Model: MultinomialNB()
Accuracy: 0.9588762701733413
F1 Score: 0.9574387747606318
Model: LinearSVC()
Accuracy: 0.9689462395836996
F1 Score: 0.96821193495038


In [32]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
        'penalty': ['l2'],
        'loss': ['hinge', 'squared_hinge'],
        'C': np.logspace(-3, 1, 100),
        'dual': [True]
    }

random_search = RandomizedSearchCV(svm, param_distributions=param_dist, n_iter=10, cv=5,verbose = 2, scoring='accuracy' )


random_search.fit(x_train_final, y_train_encoded)

print(random_search.best_params_)
print(random_search.best_score_)

best_svm = random_search.best_estimator_

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END C=4.750810162102798, dual=True, loss=hinge, penalty=l2; total time=   0.5s


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END C=4.750810162102798, dual=True, loss=hinge, penalty=l2; total time=   0.6s


...

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END C=2.9836472402833403, dual=True, loss=hinge, penalty=l2; total time=   0.7s
[CV] END C=2.9836472402833403, dual=True, loss=hinge, penalty=l2; total time=   0.5s
[CV] END C=2.9836472402833403, dual=True, loss=hinge, penalty=l2; total time=   0.5s
[CV] END C=0.15199110829529347, dual=True, loss=hinge, penalty=l2; total time=   0.3s
[CV] END C=0.15199110829529347, dual=True, loss=hinge, penalty=l2; total time=   0.3s


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END C=0.15199110829529347, dual=True, loss=hinge, penalty=l2; total time=   0.4s
[CV] END C=0.15199110829529347, dual=True, loss=hinge, penalty=l2; total time=   0.3s
[CV] END C=0.15199110829529347, dual=True, loss=hinge, penalty=l2; total time=   0.3s
{'penalty': 'l2', 'loss': 'hinge', 'dual': True, 'C': np.float64(2.9836472402833403)}
0.9672690833655638


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [33]:
from sklearn.metrics import classification_report

final_svm = LinearSVC(random_state=42)

final_svm.fit(x_train_final, y_train_encoded)

y_pred_test = final_svm.predict(x_test_final)
print(classification_report(y_test_encoded, y_pred_test))




              precision    recall  f1-score   support

           0       0.98      0.97      0.98        67
           1       0.96      1.00      0.98        55
           2       0.98      0.95      0.96        55
           3       0.97      1.00      0.99        69
           4       0.96      0.94      0.95        52

    accuracy                           0.97       298
   macro avg       0.97      0.97      0.97       298
weighted avg       0.97      0.97      0.97       298



In [34]:
import joblib

joblib.dump(final_svm, '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/svm_model.pkl')

['/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/svm_model.pkl']

In [35]:
joblib.dump(final_svm, '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/svm_model.pkl')
joblib.dump(label_encoder, '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/label_encoder.pkl')
joblib.dump(tfidf_vectorizer, '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/tfidf_vectorizer.pkl')
joblib.dump(x_train_final.columns.tolist(), '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/feature_columns.pkl')


['/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/feature_columns.pkl']